# IGDB 001: Steam ↔ IGDB join + coverage EDA

**IGDB join and coverage EDA for recommender v2** — v1 (2026-06-17)

Decision source: [`docs/plans/recommender_v2_questionnaire.md`](../../docs/plans/recommender_v2_questionnaire.md) § E, H, I.

# Executive Summary

**Question:**  
Can we join our Steam game catalog (and `val_dev_12k_v1` eval anchors) to IGDB with enough **summary** and **metadata tag** coverage to proceed with v2 ranker spikes (V2a metadata, V2b summary)?

**Result:**  
_Update after running the Evaluation section below._ Key numbers to record:
- Overall Steam → IGDB match rate (`join_report.match_rate`)
- Eval cohort query-app join rate (`join_report.eval_cohort_join_rate`)
- Field coverage on joined rows (`join_report.field_coverage_on_joined`)

**Recommendation / Decision:**  
_Update after run._ Per v2 questionnaire § E: **no hard coverage gate** — proceed to V2a-query spike if join + field coverage are usable; otherwise refine join (manual overrides / full name fallback) before ranker work.

# Business Context

**Why are we doing this?**  
v1 shipped `two_tower_v1 @100 → D1 @10`. v2 adds **rank-only** signals on frozen pools: IGDB **metadata overlap** (V2a) and **summary similarity** (V2b). Before building rankers, we need a reliable Steam `app_id` → IGDB row mapping and to know which IGDB fields are populated on our catalog.

**Expected Impact**  
- Unblocks v2 spike order: **V2a-query → V2b → V2a-history → V2c → V2d**
- Surfaces join gaps early (manual override list, generated fallbacks per § E)
- Informs whether summary-only vs metadata-only paths are viable on real catalog coverage

# Research Question

**Research Question:**  
What is the Steam `app_id` → IGDB match rate on our indexed catalog, what join method works best (`external_games` vs name fallback), and what fraction of joined games have usable **summary**, **genres**, **themes**, **keywords**, **game_modes**, **player_perspectives**, and **franchises** for v2 reranking?

# Hypothesis

**Hypothesis:**  
IGDB `external_games` (Steam uid) will match the majority of our `recs_002` catalog; joined rows will have sufficient tag metadata for V2a and summary text for V2b on most eval-cohort query games.

**Success Criteria:**  
- Auth + API smoke test passes (known Steam app resolves to IGDB row)
- Join report written with overall + eval-cohort match rates
- Field coverage reported for **all** fetched IGDB game fields (full `/v4/games` list)
- Unresolved sample exported for manual review (questionnaire § I: overrides if name match fails)
- **No hard gate** on coverage (questionnaire § E) — report slices and decide in Key Findings

# Definitions

| Term | Definition | Notes |
|------|------------|-------|
| `app_id` | Steam application id in our game index | Primary key on Steam side |
| `igdb_game_id` | IGDB internal game id | Primary key on IGDB side |
| `external_games` | IGDB endpoint mapping external store ids → IGDB game | Steam `external_game_source = 1` |
| Join pass 1 | `external_games` lookup by Steam uid | Preferred join (questionnaire § I1) |
| Join pass 2 | IGDB `search` + normalized title exact match | Fallback (questionnaire § I2); capped in this EDA |
| Field coverage | % of **joined** rows with non-empty summary or tag list | Computed on matched rows only |
| Eval cohort join rate | Match rate for `query_app_id` in `val_dev_12k_v1` | Subset that matters for offline eval |
| V2a / V2b | v2 ranker spikes | V2a = metadata; V2b = summary sim (see v2 questionnaire § J) |

# Data Sources

## Data Source 1 — Steam game catalog

**Source Name:**  
Game profile embedding index (`recs_002` output)

**Location:**  
`artifacts/recs/embeddings/game_profile/default/game_profile_embedding_index.parquet` (legacy fallback: `artifacts/recs/game_profile_embedding_index.parquet`)

**Population Covered:**  
All games in the shipped content retrieval index (one row per `app_id`)

**Filters Applied:**  
Drop null `app_id` / name; dedupe on `app_id`

**Known Limitations:**  
Catalog = games with profile embeddings, not all Steam titles

---

## Data Source 2 — Eval cohort anchors

**Source Name:**  
Frozen val eval examples

**Location:**  
`artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet`

**Population Covered:**  
Unique `query_app_id` values in val cohort (12.5k examples)

**Filters Applied:**  
None beyond dedupe

**Known Limitations:**  
Optional — skipped if parquet missing; eval join rate not reported

---

## Data Source 3 — IGDB API

**Source Name:**  
IGDB v4 API (Twitch OAuth)

**Location:**  
API base: `https://api.igdb.com/v4/` — credentials in repo-root `.env`  
API docs: [https://api-docs.igdb.com/](https://api-docs.igdb.com/) (Apicalypse query language, endpoints, fields)

**Population Covered:**  
Games reachable via `external_games` batch lookup + capped name-search fallback

**Filters Applied:**  
Steam `external_game_source = 1` for pass 1; `MAX_NAME_LOOKUPS` cap on pass 2

**Known Limitations:**  
Rate limit (~4 req/s); name search ambiguous for re-releases. Cached by pipeline job under `artifacts/igdb/`.

# Design / Process

**Methodology**  
Exploratory coverage audit on pipeline-produced join artifacts. No model training. Fetch/join runs via `scripts/recs_job_igdb_games.py`; this notebook loads results and reports coverage.

**Analysis Approach**  
1. Run pipeline (or set `REFRESH_IGDB=True` in Setup): `python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json`
2. Load `igdb_games.parquet` + `igdb_join_report.json`
3. Optional smoke-test Twitch OAuth + IGDB API (Counter-Strike / app `730`)
4. Compute / display match rates and per-field coverage; flag unresolved titles for manual overrides

**IGDB `/v4/games` field policy**  
Fetch the full documented games field list (plus `id` for joins). v2 rankers will use a subset later (`summary`, tag dimensions); this pull archives everything for coverage EDA and future spikes.

Join uses `/v4/external_games` separately — the nested `external_games` field on game rows is also stored for cross-check.

# Decision Log

| Decision | Reason | Alternatives Considered | Impact |
|----------|--------|-------------------------|--------|
| Join pass 1 = `external_games` | IGDB-native Steam uid mapping (questionnaire § I1) | Name-only join | Highest precision when uid exists |
| Join pass 2 = normalized title search | Recover orphans when external id missing (§ I2) | Fuzzy match library, manual-only | Conservative exact-normalized match; capped API cost |
| Static API pull in notebook | Questionnaire § H: EDA first, then pipeline | Pre-built third-party dump | Live coverage on *our* catalog |
| `MAX_NAME_LOOKUPS = 200` | Limit EDA runtime / rate limits | Full-catalog name fallback | Understates pass-2 join until cap raised |
| No hard coverage gate | Questionnaire § E | Block v2 until ≥ X% | Report slices; decide in findings |

# Evaluation Outputs / Artifacts

| File / Artifact Name | Location | Description | How it was generated | How it should be used or interpreted |
|----------------------|----------|-------------|------------------------|--------------------------------------|
| `igdb_games.parquet` | `artifacts/igdb/` | Joined Steam + IGDB fields per `app_id` | This notebook, final write cell | Input for v2 ranker feature builders |
| `igdb_join_report.json` | `artifacts/igdb/` | Match rates, method counts, field coverage | Coverage evaluation cell | Go/no-go for v2 spikes; override prioritization |
| `meta.json` | `artifacts/igdb/` | Pull timestamp, batch sizes, API config | Final write cell | Reproducibility / audit trail |

# Notebook Roadmap

1. Setup (paths; optional in-notebook refresh via `REFRESH_IGDB`)
2. Load pipeline artifacts (`igdb_games.parquet`, join report) or re-fetch
3. Validate catalog quality; optional API smoke test
4. Coverage evaluation + unresolved sample
5. Key findings / recommendation (manual update)

**Pipeline command:** `python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json`

# Analysis

_Code cells below implement the roadmap. Run top-to-bottom._

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.igdb import V2_CORE_FIELDS, IgdbGamesJobConfig
from steam_review_ml.igdb.constants import IGDB_GAMES_PARQUET, STEAM_JOIN_COLS
from steam_review_ml.igdb.fetch import (
    fetch_and_join_igdb_games,
    load_eval_query_app_ids,
    load_steam_catalog,
    resolve_game_index_path,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
OUT_DIR = REPO_ROOT / "artifacts/igdb"
EVAL_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"

# Set True to re-fetch from IGDB in-notebook (normally use the pipeline script).
REFRESH_IGDB = False

print(f"REPO_ROOT={REPO_ROOT}")
print(f"OUT_DIR={OUT_DIR}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations
OUT_DIR=/home/ryanr/workspace/steam_recommendations/artifacts/igdb


In [2]:
if REFRESH_IGDB:
    job_config = IgdbGamesJobConfig(
        repo_root=REPO_ROOT,
        output_dir=OUT_DIR,
        eval_parquet_path=EVAL_PARQUET if EVAL_PARQUET.is_file() else None,
    )
    steam_catalog, joined, join_report, meta = fetch_and_join_igdb_games(job_config)
else:
    parquet_path = OUT_DIR / IGDB_GAMES_PARQUET
    report_path = OUT_DIR / "igdb_join_report.json"
    meta_path = OUT_DIR / "meta.json"
    if not parquet_path.is_file() or not report_path.is_file():
        raise FileNotFoundError(
            "Missing IGDB artifacts. Run:\n"
            "  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json"
        )
    joined = pd.read_parquet(parquet_path)
    join_report = json.loads(report_path.read_text(encoding="utf-8"))
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}
    index_path = Path(join_report.get("index_path", resolve_game_index_path(REPO_ROOT)))
    steam_catalog = load_steam_catalog(index_path)

join_map = joined[["app_id", "igdb_game_id", "join_method"]].drop_duplicates()
index_path = Path(join_report.get("index_path", resolve_game_index_path(REPO_ROOT)))
MAX_NAME_LOOKUPS = int(join_report.get("name_fallback_cap", meta.get("max_name_lookups", 200)))
eval_app_ids = load_eval_query_app_ids(EVAL_PARQUET if EVAL_PARQUET.is_file() else None)

print(f"loaded rows={len(joined)} match_rate={join_report['match_rate']:.1%}")
print(f"join methods: {join_report.get('join_method_counts')}")

loaded rows=315 match_rate=100.0%
join methods: {'external_games': 314, 'manual_mock': 1}


## Load Data

In [3]:
print(f"index_path={index_path}")
print(f"steam_catalog_n={len(steam_catalog)}")
print(f"eval_cohort_unique_query_app_ids={len(eval_app_ids)}")
display(steam_catalog.head())

index_path=/home/ryanr/workspace/steam_recommendations/artifacts/recs/embeddings/game_profile/default/game_profile_embedding_index.parquet
steam_catalog_n=315
eval_cohort_unique_query_app_ids=297


,app_id,app_name,app_name_norm
0,70,Half-Life,half life
1,240,Counter-Strike: Source,counter strike source
2,420,Half-Life 2: Episode Two,half life 2 episode two
3,620,Portal 2,portal 2
4,2870,X Rebirth,x rebirth


## Validate Data Quality

In [4]:
from steam_review_ml.igdb.client import IGDBClient
from steam_review_ml.igdb.constants import STEAM_EXTERNAL_GAME_SOURCE
from steam_review_ml.igdb.fetch import load_twitch_credentials

print("Catalog null rates:")
display(steam_catalog[["app_id", "app_name"]].isna().mean().to_frame("null_rate"))
print(f"duplicate app_id rows: {steam_catalog['app_id'].duplicated().sum()}")

# Optional smoke test: Counter-Strike (Steam app 730) via external_games
client_id, client_secret = load_twitch_credentials(repo_root=REPO_ROOT)
client = IGDBClient(client_id, client_secret)
smoke = client.post(
    "external_games",
    f'fields game, uid, external_game_source; where external_game_source = {STEAM_EXTERNAL_GAME_SOURCE} & uid = "730"; limit 1;',
)
print("smoke external_games:", smoke)
if not smoke:
    raise RuntimeError("IGDB smoke test failed — check Twitch credentials and API access.")

igdb_id = int(smoke[0]["game"])
smoke_detail = client.post(
    "games",
    "fields id,name,summary,genres.name,themes.name,keywords.name; "
    f"where id = {igdb_id}; limit 1;",
)
print("smoke game detail:", smoke_detail[0] if smoke_detail else None)

Catalog null rates:


,null_rate
app_id,0.0
app_name,0.0


duplicate app_id rows: 0
smoke external_games: [{'id': 15147, 'game': 242408, 'uid': '730', 'external_game_source': 1}]
smoke game detail: {'id': 242408, 'genres': [{'id': 5, 'name': 'Shooter'}, {'id': 24, 'name': 'Tactical'}], 'keywords': [{'id': 4358, 'name': 'loot boxes'}, {'id': 5466, 'name': 'terrorists'}, {'id': 17391, 'name': 'competitve'}, {'id': 41334, 'name': 'bombs'}], 'name': 'Counter-Strike 2', 'summary': 'For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.\n\nA free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools.', 'themes': [{'id': 1, 'name': 'Action'}, {'id': 39, 'name': 'Warfare'}]}

## Core Analysis — Joined IGDB dataset

Loaded from pipeline artifacts (or refreshed in Setup). Join pass 1 = `external_games`; pass 2 = name-search fallback (capped by `max_name_lookups` in config).

In [5]:
print(f"joined columns: {len(joined.columns)} (full IGDB games payload + Steam join cols)")
display(join_map["join_method"].value_counts().to_frame("count"))
display(joined[["app_id", "app_name", "igdb_name", "summary", "genres"]].head(10))

joined columns: 22 (full IGDB games payload + Steam join cols)


,count
join_method,
external_games,314
manual_mock,1


,app_id,app_name,igdb_name,summary,genres
0,753420,Dungreed,Dungreed,Dungreed is 2D side-scrolling action game with...,"[8, 31, 32]"
1,646910,The Crew 2,The Crew 2,The newest iteration in the revolutionary fran...,[10]
2,512900,Streets of Rogue,Streets of Rogue,Streets of Rogue is a top-down rogue-lite with...,"[5, 12, 25, 31, 32]"
3,637090,BATTLETECH,BattleTech,BattleTech is a turn-based tactical 'Mech comb...,"[15, 16, 31]"
4,613830,CHRONO TRIGGER,Chrono Trigger,"As the definitive version of Chrono Trigger, n...","[12, 31]"
5,748490,The Legend of Heroes: Trails of Cold Steel II,The Legend of Heroes: Trails of Cold Steel II,The Legend of Heroes: Trails of Cold Steel II ...,"[12, 15, 16, 31]"
6,420530,OneShot,OneShot,OneShot is a surreal top down puzzle/adventure...,"[9, 12, 31, 32]"
7,753650,Due Process,Due Process,A PVP tactical FPS about planning & execution....,"[5, 32]"
8,825630,STEINS;GATE 0,Steins;Gate 0,Steins;Gate 0 is a Japanese visual novel. It i...,"[31, 34]"
9,1144400,Senren＊Banka,Senren Banka,The village of Hoori lies deep in the middle o...,"[31, 34]"


In [6]:
print("Pipeline meta.json:")
print(json.dumps(meta, indent=2))

Pipeline meta.json:
{
  "created_utc": "2026-06-19T12:13:16.176752+00:00",
  "igdb_api_base": "https://api.igdb.com/v4",
  "steam_external_game_source": 1,
  "external_batch_size": 500,
  "game_detail_batch_size": 50,
  "max_name_lookups": 200,
  "game_fields_preset": "pipeline",
  "game_fields": "id,name,age_ratings,collections,franchises,game_engines,game_modes,game_type,genres,involved_companies,keywords,multiplayer_modes,player_perspectives,storyline,summary,tags,themes",
  "mock_rows_path": "/home/ryanr/workspace/steam_recommendations/configs/igdb_steam_mock_rows.json",
  "output_dir": "/home/ryanr/workspace/steam_recommendations/artifacts/igdb",
  "output_filename": "lookups/games.parquet"
}


## Evaluation

In [7]:
from datetime import datetime, timezone

def _field_populated(value: Any) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, dict, set, tuple)):
        return len(value) > 0
    return True


catalog_n = len(steam_catalog)
joined_n = len(joined)
unresolved_n = catalog_n - joined_n

igdb_field_cols = [c for c in joined.columns if c not in STEAM_JOIN_COLS]
field_coverage = {
    f"{col}_pct": float(joined[col].map(_field_populated).mean()) if joined_n else 0.0
    for col in igdb_field_cols
}
v2_core_coverage = {k: field_coverage.get(f"{k}_pct", 0.0) for k in V2_CORE_FIELDS if f"{k}_pct" in field_coverage}

join_method_counts = join_map["join_method"].value_counts().to_dict()

eval_join_rate = None
if eval_app_ids:
    eval_joined = len(set(joined["app_id"].tolist()) & eval_app_ids)
    eval_join_rate = eval_joined / len(eval_app_ids)

join_report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "steam_catalog_n": catalog_n,
    "joined_n": joined_n,
    "unresolved_n": unresolved_n,
    "match_rate": joined_n / catalog_n if catalog_n else 0.0,
    "join_method_counts": join_method_counts,
    "field_coverage_on_joined": field_coverage,
    "v2_core_field_coverage": v2_core_coverage,
    "igdb_field_count": len(igdb_field_cols),
    "eval_cohort_unique_query_app_ids": len(eval_app_ids),
    "eval_cohort_join_rate": eval_join_rate,
    "name_fallback_cap": MAX_NAME_LOOKUPS,
    "index_path": str(index_path),
}

coverage_df = pd.DataFrame(
    [{"field": k.replace("_pct", ""), "coverage_pct": round(v * 100, 2)} for k, v in field_coverage.items()]
).sort_values("coverage_pct", ascending=False)

print("V2 core field coverage:")
display(
    pd.DataFrame(
        [{"field": k, "coverage_pct": round(v * 100, 2)} for k, v in v2_core_coverage.items()]
    ).sort_values("coverage_pct", ascending=False)
)
print("\nAll IGDB fields (top 20 by coverage):")
print(json.dumps(join_report, indent=2))
display(coverage_df)

if unresolved_n:
    unresolved_df = steam_catalog.loc[~steam_catalog["app_id"].isin(join_map["app_id"])].head(20)
    print("\nSample unresolved Steam titles (candidates for manual overrides):")
    display(unresolved_df)

V2 core field coverage:


,field,coverage_pct
0,summary,100.00
1,genres,100.00
4,game_modes,100.00
2,themes,98.41
5,player_perspectives,96.19
3,keywords,91.43
6,franchises,30.16



All IGDB fields (top 20 by coverage):
{
  "created_utc": "2026-06-19T12:17:44.363661+00:00",
  "steam_catalog_n": 315,
  "joined_n": 315,
  "unresolved_n": 0,
  "match_rate": 1.0,
  "join_method_counts": {
    "external_games": 314,
    "manual_mock": 1
  },
  "field_coverage_on_joined": {
    "age_ratings_pct": 0.8539682539682539,
    "franchises_pct": 0.30158730158730157,
    "game_engines_pct": 0.7841269841269841,
    "game_modes_pct": 1.0,
    "genres_pct": 1.0,
    "involved_companies_pct": 0.9936507936507937,
    "keywords_pct": 0.9142857142857143,
    "multiplayer_modes_pct": 0.4158730158730159,
    "player_perspectives_pct": 0.9619047619047619,
    "storyline_pct": 0.4984126984126984,
    "summary_pct": 1.0,
    "tags_pct": 1.0,
    "themes_pct": 0.9841269841269841,
    "collections_pct": 0.653968253968254,
    "game_type_pct": 1.0,
    "summary__use_pct": 1.0,
    "storyline__use_pct": 1.0
  },
  "v2_core_field_coverage": {
    "summary": 1.0,
    "genres": 1.0,
    "themes":

,field,coverage_pct
4,genres,100.00
14,game_type,100.00
3,game_modes,100.00
11,tags,100.00
16,storyline__use,100.00
10,summary,100.00
15,summary__use,100.00
5,involved_companies,99.37
12,themes,98.41
8,player_perspectives,96.19


In [8]:
# Get list of fields with more than X% coverage_pct (set threshold as needed, e.g., 90)
COVERAGE_THRESHOLD = 95.0  # percent

high_coverage_fields = coverage_df[coverage_df["coverage_pct"] > COVERAGE_THRESHOLD]["field"].tolist()
print(f"Fields with coverage > {COVERAGE_THRESHOLD}%:")
print(high_coverage_fields)

Fields with coverage > 95.0%:
['genres', 'game_type', 'game_modes', 'tags', 'storyline__use', 'summary', 'summary__use', 'involved_companies', 'themes', 'player_perspectives']


In [9]:
print(f"Artifacts at {OUT_DIR}")
print(f"  igdb_games.parquet ({len(joined)} rows)")
print(f"  igdb_join_report.json")
print(f"  meta.json")
print("\nRefresh via:")
print("  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json")

Artifacts at /home/ryanr/workspace/steam_recommendations/artifacts/igdb
  igdb_games.parquet (315 rows)
  igdb_join_report.json
  meta.json

Refresh via:
  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json


In [10]:
# Auto-draft Executive Summary bullets from join_report (copy into top section after run)
fc = join_report["field_coverage_on_joined"]
eval_rate_txt = (
    f"{join_report['eval_cohort_join_rate']:.1%}"
    if join_report["eval_cohort_join_rate"] is not None
    else "n/a (eval parquet missing)"
)
summary_md = f"""
**Result (auto-draft):**
- Overall match rate: **{join_report['match_rate']:.1%}** ({join_report['joined_n']}/{join_report['steam_catalog_n']})
- Eval cohort query-app join rate: **{eval_rate_txt}**
- Join methods: {join_report['join_method_counts']}
- Summary coverage (joined): **{fc['summary_pct']:.1%}** | Genres: **{fc['genres_pct']:.1%}** | Themes: **{fc['themes_pct']:.1%}**

**Recommendation (auto-draft):**
- If eval join rate and tag coverage are usable → proceed to **V2a-query** metadata spike.
- If summary coverage is strong → schedule **V2b** summary spike next.
- If match rate is low → raise `MAX_NAME_LOOKUPS` or add `configs/igdb_steam_overrides.csv` for unresolved high-traffic titles.
"""
display(Markdown(summary_md))


**Result (auto-draft):**
- Overall match rate: **100.0%** (315/315)
- Eval cohort query-app join rate: **100.0%**
- Join methods: {'external_games': 314, 'manual_mock': 1}
- Summary coverage (joined): **100.0%** | Genres: **100.0%** | Themes: **98.4%**

**Recommendation (auto-draft):**
- If eval join rate and tag coverage are usable → proceed to **V2a-query** metadata spike.
- If summary coverage is strong → schedule **V2b** summary spike next.
- If match rate is low → raise `MAX_NAME_LOOKUPS` or add `configs/igdb_steam_overrides.csv` for unresolved high-traffic titles.


# Key Findings

_Update after run. Use the auto-draft cell above + `igdb_join_report.json`._

**Finding 1:**  
[Overall Steam → IGDB match rate and primary join method (`external_games` vs `name_search`).]

**Finding 2:**  
[Eval cohort (`val_dev_12k_v1`) query-app join rate — does offline eval coverage look sufficient?]

**Finding 3:**  
[Field coverage on joined rows: summary, genres, themes, keywords — which v2 signals are viable?]

**Unexpected Results:**  
[Orphans, ambiguous name matches, missing summaries on popular titles, API limits hit, etc.]

# Recommendation / Next Steps

**Recommended Action:**  
1. If join + coverage acceptable → start **V2a-query** (metadata overlap vs `query_app_id`) on frozen pools  
2. Then **V2b** (summary sim) if summary coverage supports it  
3. Persist this pull as the static cache under `artifacts/igdb/` for ranker notebooks (questionnaire § H)

**Risks:**  
- Name fallback cap understates pass-2 matches until raised  
- IGDB API rate limits / token expiry on long runs  
- Missing IGDB rows for eval anchors → neutral fallback or Cursor-generated metadata (questionnaire § E)

**Follow-up Analyses:**  
- Full-catalog name fallback (raise `MAX_NAME_LOOKUPS`)  
- Manual override table for high-impact unresolved titles  
- Slice coverage: eval cohort vs long-tail catalog

**Open Questions:**  
- Is Jaccard on tags sufficient (§ B3) or do we need embedded tag strings?  
- Do unresolved eval query games need generated metadata before V2a spike?